# Synthetic Forcing for Advection-Diffusion Comparison

This notebook generates synthetic forcing data to compare advection-diffusion between seapodym-lmtl and the new transport model.

**Configuration:**
- Domain: 360°E × 180°N at 1° resolution
- Time: 5 years (2000-2005), daily
- Depth: 1 level
- Advection: Single centered vortex (stationary)
- Mask: 4 islands (~5-10% domain each)
- Variables: current_u, current_v, temperature=0, primary_production=0

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
export_folder = "./LMTL_Synthetic/"

In [ ]:
(Path(export_folder) / "data/").mkdir(exist_ok=True, parents=True)
(Path(export_folder) / "output/").mkdir(exist_ok=True, parents=True)

## Define Domain

In [ ]:
# Spatial domain: 1° resolution
longitude = np.arange(0.5, 360.5, 1.0, dtype=np.float32)  # 360 points
latitude = np.arange(-69.5, 69.5, 1.0, dtype=np.float32)  # 180 points
depth = np.array([1], dtype=np.float32)  # Single level, ADMB 1-based indexing

# Temporal domain: 5 years daily
time = pd.date_range("2000-01-01", "2004-12-31", freq="D")

print(
    f"Domain: {len(longitude)}°lon × {len(latitude)}°lat × {len(depth)} depth × {len(time)} timesteps"
)

## Generate Velocity Field

Creating a uniform eastward current for easy visualization of advection.

Current configuration:
- u = constant > 0 (eastward)
- v = 0 (no meridional flow)

CFL condition: u × Δt / Δx < 1
- At 1° resolution and 1 day timestep
- Δx ≈ 111 km, Δt = 86400s (1 day)
- u_max < 1.28 m/s for stability

In [ ]:
# Current parameters
x0, y0 = 180.0, 0.0  # Reference center position (for blob)

# Uniform eastward current
# CFL constraint: u × dt / dx < 1
# With dt = 86400s, dx ≈ 111km = 111000m
# u_max < 111000 / 86400 ≈ 1.28 m/s
# We'll use u = 1.0 m/s for safety

u_constant = 2.0  # m/s - eastward velocity

# Create meshgrid (needed for blob calculation later)
lon_2d, lat_2d = np.meshgrid(longitude, latitude)

# Create uniform velocity fields
u_field = np.full_like(lon_2d, u_constant, dtype=np.float32)
v_field = np.zeros_like(lon_2d, dtype=np.float32)

# Calculate distances from center (for blob later)
dx_vortex = lon_2d - x0
dx_vortex = np.where(dx_vortex > 180, dx_vortex - 360, dx_vortex)
dx_vortex = np.where(dx_vortex < -180, dx_vortex + 360, dx_vortex)
dy_vortex = lat_2d - y0

# Calculate CFL
velocity_mag = np.sqrt(u_field**2 + v_field**2)
v_max = velocity_mag.max()
dx_meters = 111000.0  # Approximate meters per degree at equator
dt_seconds = 3 * 60 * 60  # 1 day in seconds
cfl_max = v_max * dt_seconds / dx_meters

print(f"Velocity: u = {u_constant:.3f} m/s (eastward), v = 0.0 m/s")
print(f"Max speed: {v_max:.3f} m/s")
print(f"CFL number (advection): {cfl_max:.3f} (should be < 1.0)")

if cfl_max < 1.0:
    print("✅ CFL condition satisfied")
else:
    print("⚠️  CFL condition violated!")

In [ ]:
# Visualize vortex
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot u component
im0 = axes[0].contourf(lon_2d, lat_2d, u_field, levels=20, cmap="RdBu_r")
axes[0].set_title("Zonal velocity (u)")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
plt.colorbar(im0, ax=axes[0], label="m/s")

# Plot v component
im1 = axes[1].contourf(lon_2d, lat_2d, v_field, levels=20, cmap="RdBu_r")
axes[1].set_title("Meridional velocity (v)")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
plt.colorbar(im1, ax=axes[1], label="m/s")

plt.tight_layout()
plt.show()

In [ ]:
# Plot streamlines
fig, ax = plt.subplots(figsize=(12, 6))
speed = np.sqrt(u_field**2 + v_field**2)
strm = ax.streamplot(
    lon_2d, lat_2d, u_field, v_field, color=speed, cmap="viridis", density=1.5, linewidth=1
)
ax.set_title("Vortex Flow Field")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.colorbar(strm.lines, ax=ax, label="Speed (m/s)")
plt.show()

## Generate Islands Mask

Creating 4 circular islands positioned symmetrically around the vortex center.

In [ ]:
# Island parameters
# Target: each island ~5-10% of domain
# Total domain area = 360 × 180 = 64,800 deg²
# Each island ~3,240-6,480 deg² → radius ~15-25°

# island_radius = 20.0  # degrees
# island_distance = 75.0  # distance from vortex center

# Island centers (positioned at cardinal directions from vortex center)
# islands = [
#     (x0 + island_distance, y0),  # East
#     (x0 - island_distance, y0),  # West
#     (x0, y0 + island_distance),  # North
#     (x0, y0 - island_distance),  # South
# ]

# Create mask (1 = ocean, 0 = land/island)
mask = np.ones_like(lon_2d, dtype=np.int32)

# for island_lon, island_lat in islands:
#     # Calculate distance from island center
#     dx_island = lon_2d - island_lon
#     dx_island = np.where(dx_island > 180, dx_island - 360, dx_island)
#     dx_island = np.where(dx_island < -180, dx_island + 360, dx_island)
#     dy_island = lat_2d - island_lat
#     r_island = np.sqrt(dx_island**2 + dy_island**2)

#     # Mark island cells
#     mask[r_island < island_radius] = 0

ocean_pct = (mask.sum() / mask.size) * 100
land_pct = 100 - ocean_pct
print(f"Ocean: {ocean_pct:.1f}%, Land: {land_pct:.1f}%")
print(f"Each island represents ~{land_pct / 4:.1f}% of total domain")

In [ ]:
# Visualize mask with velocity field
fig, ax = plt.subplots(figsize=(12, 6))

# Plot mask
im = ax.contourf(lon_2d, lat_2d, mask, levels=[0, 0.5, 1], colors=["#8B4513", "#4169E1"], alpha=0.6)

# Overlay streamlines
# Only plot where mask == 1 (ocean)
u_masked = np.where(mask == 1, u_field, np.nan)
v_masked = np.where(mask == 1, v_field, np.nan)
speed_masked = np.sqrt(u_masked**2 + v_masked**2)

strm = ax.streamplot(
    lon_2d, lat_2d, u_masked, v_masked, color=speed_masked, cmap="viridis", density=1.2, linewidth=1
)

ax.set_title("Domain Mask with Vortex Flow (Brown=Land, Blue=Ocean)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.colorbar(strm.lines, ax=ax, label="Speed (m/s)")
plt.show()

## Create Dataset

In [ ]:
# Create dataset with all variables
data = xr.Dataset(
    {
        "current_u": (
            ["time", "depth", "latitude", "longitude"],
            np.broadcast_to(
                u_field[np.newaxis, np.newaxis, :, :],
                (len(time), len(depth), len(latitude), len(longitude)),
            ),
        ),
        "current_v": (
            ["time", "depth", "latitude", "longitude"],
            np.broadcast_to(
                v_field[np.newaxis, np.newaxis, :, :],
                (len(time), len(depth), len(latitude), len(longitude)),
            ),
        ),
        "temperature": (
            ["time", "depth", "latitude", "longitude"],
            np.zeros((len(time), len(depth), len(latitude), len(longitude)), dtype=np.float32),
        ),
        "primary_production": (
            ["time", "latitude", "longitude"],
            np.zeros((len(time), len(latitude), len(longitude)), dtype=np.float32),
        ),
    },
    coords={
        "longitude": longitude,
        "latitude": latitude,
        "depth": depth,
        "time": time,
    },
)

# Set coordinate attributes
data.latitude.attrs.update({"standard_name": "latitude"})
data.longitude.attrs.update({"standard_name": "longitude"})
data.depth.attrs.update({"standard_name": "depth"})
data.time.attrs.update({"standard_name": "time"})

# Set variable attributes
data.temperature.attrs = {"standard_name": "temperature"}
data.primary_production.attrs = {"standard_name": "net primary production"}
data.current_u.attrs = {"standard_name": "longitudinal current"}
data.current_v.attrs = {"standard_name": "latitudinal current"}

# data = data.transpose("time", "latitude", "longitude", "depth")

data

## Create and Export Mask

In [ ]:
# Create mask DataArray
mask_da = xr.DataArray(
    mask,
    dims=["latitude", "longitude"],
    coords={"latitude": latitude, "longitude": longitude},
    name="mask",
)
mask_da.attrs = {"standard_name": "mask"}

# Set coordinate attributes
mask_da.latitude.attrs.update({"standard_name": "latitude"})
mask_da.longitude.attrs.update({"standard_name": "longitude"})

mask_da

In [ ]:
# Export mask
mask_da.to_netcdf(f"{export_folder}/mask.nc")
print(f"Mask exported to {export_folder}/mask.nc")

# Verify
xr.load_dataset(f"{export_folder}/mask.nc")

## Export Daily Files

Exporting one NetCDF file per day, following the same structure as the real forcing data.

In [ ]:
# Export daily files
for i, date in enumerate(data.time.values):
    date_as_string = pd.to_datetime(str(date)).strftime("%Y%m%d")
    data.sel(time=[date])[
        ["current_u", "current_v", "temperature", "primary_production"]
    ].to_netcdf(f"{export_folder}/data/data_{date_as_string}.nc")

    if (i + 1) % 365 == 0:
        print(f"Exported {i + 1}/{len(data.time)} files...")

print(f"\nAll {len(data.time)} daily files exported to {export_folder}/data/")

In [ ]:
# Biomass blob parameters
biomass_max = 1.0  # maximum biomass value
blob_radius = 20.0  # degrees - similar to island size

# Calculate distance from center for Gaussian blob
# (reusing the dx_vortex and dy_vortex from vortex calculation)
r_blob = np.sqrt(dx_vortex**2 + dy_vortex**2)
biomass_field = biomass_max * np.exp(-(r_blob**2) / (blob_radius**2))

# Apply mask: biomass = 0 on land
# biomass_field = biomass_field * mask

# Convert to float32
biomass_field = biomass_field.astype(np.float32)

print(f"Biomass range: [{biomass_field.min():.3f}, {biomass_field.max():.3f}]")
print(f"Total biomass: {biomass_field.sum():.1f}")

In [ ]:
# Create initial state dataset
initial_time = pd.to_datetime("2000-01-01")

biomass_initial = xr.DataArray(
    biomass_field[np.newaxis, :, :],  # Add time dimension
    dims=["time", "latitude", "longitude"],
    coords={
        "time": [initial_time],
        "latitude": latitude,
        "longitude": longitude,
    },
    name="biomass",
)

biomass_initial.attrs = {"standard_name": "biomass", "units": "arbitrary"}
biomass_initial.latitude.attrs.update({"standard_name": "latitude"})
biomass_initial.longitude.attrs.update({"standard_name": "longitude"})
biomass_initial.time.attrs.update({"standard_name": "time"})

biomass_initial

In [ ]:
# Visualize biomass blob with islands
fig, ax = plt.subplots(figsize=(12, 6))

# Plot biomass
im = ax.contourf(lon_2d, lat_2d, biomass_field, levels=20, cmap="YlOrRd")
plt.colorbar(im, ax=ax, label="Biomass")

# Overlay mask (islands)
ax.contour(lon_2d, lat_2d, mask, levels=[0.5], colors="black", linewidths=2)

# Mark island positions
# for island_lon, island_lat in islands:
#     ax.plot(
#         island_lon,
#         island_lat,
#         "ko",
#         markersize=8,
#         label="Island center" if island_lon == islands[0][0] else "",
#     )

# Mark vortex center
ax.plot(x0, y0, "b*", markersize=15, label="Vortex/Blob center")

ax.set_title("Initial Biomass State (Gaussian Blob)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.show()

In [ ]:
# Export initial biomass state
biomass_initial.to_netcdf(f"{export_folder}/initial_conditions/ZPK_D1N1_biomass_19991231.nc")
print(
    f"Initial biomass state exported to {export_folder}/initial_conditions/ZPK_D1N1_biomass_19991231.nc"
)

# Verify
xr.load_dataset(f"{export_folder}/initial_conditions/ZPK_D1N1_biomass_19991231.nc")